# CSC5035Z A2 — Experiment Pipeline
**Student:** Praise Jaravani (JRVPRA001) | **Language:** Yoruba (`yor`)

This notebook runs the complete experiment pipeline. It is structured in two passes:

| Pass | Steps | LR | Purpose |
|---|---|---|---|
| Initial run | 1–8 | 2e-5 | Establish baseline and Extension B |
| Sweep + retrain | 9–10 | 5e-5 | Identify optimal LR, retrain all models |

**The outputs from Step 10 are the final reported results.**

---

### Scripts in this project
| Script | Purpose |
|---|---|
| `config.py` | Single source of truth for all hyperparameters and paths |
| `utils.py` | Shared helpers: seed setting, LR scheduler, NER label alignment |
| `analyse_tokenizer.py` | Measures token-per-word fertility on the Yoruba corpus |
| `train_news.py` / `train_news_extb.py` | Fine-tune mmBERT-small on MasakhaNews (baseline / Ext B) |
| `train_ner.py` / `train_ner_extb.py` | Fine-tune mmBERT-small on MasakhaNER 2.0 (baseline / Ext B) |
| `extend_vocab.py` | Extension B: trains Yoruba BPE vocab, extends tokenizer + model embeddings |
| `sweep_lr.py` | Sweeps LR in {1e-5, 2e-5, 3e-5, 5e-5} on train/val only — test set never touched |
| `evaluate.py` | Standalone test-set evaluation for any saved checkpoint |

**After a Colab disconnect:** re-run Cell 1 only, then resume from where you left off.

In [2]:
# Cell 1 — Set up working directory (works on Google Colab and locally)
import os, pathlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_PATH = '/content/drive/MyDrive/csc5035z-a2'
except ImportError:
    # Running locally — assumes Jupyter was launched from the project root
    PROJECT_PATH = str(pathlib.Path().resolve())

os.chdir(PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')
print('Files found:', os.listdir('.'))

In [4]:
# Cell 2 — Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU — go to Runtime > Change runtime type > T4 GPU')

Wed May 20 18:29:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
# Cell 3 — Install dependencies
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)
print('Install complete' if result.returncode == 0 else result.stderr)

Install complete


In [6]:
# Cell 4 — Verify imports
import torch, transformers, datasets
print(f'PyTorch:       {torch.__version__}')
print(f'Transformers:  {transformers.__version__}')
print(f'Datasets:      {datasets.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:           {torch.cuda.get_device_name(0)}')

PyTorch:       2.10.0+cu128
Transformers:  5.0.0
Datasets:      3.6.0
GPU available: True
GPU:           Tesla T4


---
## Step 1: Tokenizer fertility analysis — baseline

**Script:** `analyse_tokenizer.py`

Measures how the stock mmBERT-small tokenizer handles Yoruba text by computing the token-per-word ratio across the full training corpus (MasakhaNews + MasakhaNER, 8,309 sentences, ~198k words).

Yoruba uses stacked Unicode diacritics that standard multilingual tokenizers split into individual combining characters, producing very high fertility.

**What to look for in the output:**
- `Mean toks/word` — average tokens per whitespace-split word; higher = worse
- `% single-token` — fraction of words tokenised as one unit; lower = worse
- `Diacritic word probe` — qualitative check on 10 common Yoruba words with diacritics

**Output:** `results/tokenizer_analysis_baseline.json`

In [ ]:
!python analyse_tokenizer.py


Loading tokenizer from: jhu-clsp/mmBERT-small
config.json: 1.19kB [00:00, 3.67MB/s]
tokenizer_config.json: 46.4kB [00:00, 115MB/s]
tokenizer.json: 100% 17.5M/17.5M [00:01<00:00, 13.5MB/s]
special_tokens_map.json: 100% 636/636 [00:00<00:00, 4.50MB/s]
Loading datasets...
README.md: 10.7kB [00:00, 27.0MB/s]
train.tsv: 3.78MB [00:00, 105MB/s]
dev.tsv: 517kB [00:00, 93.4MB/s]
test.tsv: 1.07MB [00:00, 123MB/s]
Generating train split: 100% 1433/1433 [00:00<00:00, 14700.44 examples/s]
Generating validation split: 100% 206/206 [00:00<00:00, 8602.59 examples/s]
Generating test split: 100% 411/411 [00:00<00:00, 12678.60 examples/s]
README.md: 8.62kB [00:00, 14.6MB/s]
masakhaner2.py: 8.79kB [00:00, 19.9MB/s]
yor/train/0000.parquet: 100% 597k/597k [00:06<00:00, 90.3kB/s]
yor/validation/0000.parquet: 100% 75.4k/75.4k [00:01<00:00, 62.2kB/s]
yor/test/0000.parquet: 100% 147k/147k [00:02<00:00, 66.6kB/s]
Generating train split: 100% 6876/6876 [00:00<00:00, 190708.17 examples/s]
Generating validation s

---
## Step 2: Baseline training — MasakhaNews (text classification)

**Script:** `train_news.py`

Fine-tunes mmBERT-small on the MasakhaNews Yoruba subset (1,433 train / 206 val / 411 test, 5 categories). A linear classification head is added over the `[CLS]` token representation.

**Training details:**
- Loss: cross-entropy | Optimiser: AdamW | Scheduler: linear warmup (100 steps)
- Early stopping: patience = 2 epochs on validation macro-F1
- Best checkpoint saved and reloaded for test evaluation — test set never seen during training

**Note on the load report:** `UNEXPECTED` keys are decoder/MLM weights from pre-training (harmless — not needed for classification). `MISSING` keys are the new classification head being randomly initialised (expected).

**Output:** `checkpoints/news_baseline/` and `results/news_baseline.json`

In [ ]:
!python train_news.py

Using device: cuda (Tesla T4)
TRAINING — MasakhaNews  [news_baseline]
  model: jhu-clsp/mmBERT-small
  dataset: masakhane/masakhanews/yor
  max_seq_len: 128
  lr: 2e-05
  batch_size: 16
  epochs: 5
  warmup_steps: 100
  weight_decay: 0.01
  patience: 2
  seed: 42

Loading dataset ...
Labels (5): ['entertainment', 'health', 'politics', 'religion', 'sports']
Split sizes — train: 1433, val: 206, test: 411

Loading tokenizer + model from jhu-clsp/mmBERT-small ...
Loading weights: 100% 136/136 [00:00<00:00, 4889.68it/s, Materializing param=model.layers.21.mlp_norm.weight]
ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-small
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params w

---
## Step 3: Baseline training — MasakhaNER 2.0 (token classification)

**Script:** `train_ner.py`

Fine-tunes mmBERT-small on MasakhaNER 2.0 Yoruba (6,876 train / 983 val / 1,964 test). A linear head is applied to every token representation to predict BIO labels (O, B/I-PER, B/I-ORG, B/I-LOC, B/I-DATE).

**Label alignment:** The first subword of each word receives the word-level NER label; continuation subwords and special tokens receive label `-100`, which is excluded from the cross-entropy loss. This prevents the model from learning inconsistent span boundaries.

**Evaluation:** seqeval span-level F1 — exact span boundaries must match; partial credit is not given.

**Output:** `checkpoints/ner_baseline/` and `results/ner_baseline.json`

In [ ]:
!python train_ner.py

Using device: cuda (Tesla T4)
TRAINING — MasakhaNER 2.0  [ner_baseline]
  model: jhu-clsp/mmBERT-small
  dataset: masakhane/masakhaner2/yor
  max_seq_len: 128
  lr: 2e-05
  batch_size: 16
  epochs: 5
  warmup_steps: 100
  weight_decay: 0.01
  patience: 2
  seed: 42

Loading dataset ...
Labels (9): ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE']
Split sizes — train: 6876, val: 983, test: 1964

Loading tokenizer + model from jhu-clsp/mmBERT-small ...
Loading weights: 100% 136/136 [00:00<00:00, 1250.00it/s, Materializing param=model.layers.21.mlp_norm.weight]
ModernBertForTokenClassification LOAD REPORT from: jhu-clsp/mmBERT-small
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
decoder.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSI

---
## Step 4: Extension B — Vocabulary adaptation

**Script:** `extend_vocab.py`

Extends the mmBERT-small tokenizer with Yoruba-specific BPE tokens to reduce the token-per-word ratio caused by diacritic fragmentation. The pipeline has six steps:

1. **Corpus collection** — combine Yoruba train splits from both tasks (~8k sentences)
2. **BPE training** — HuggingFace `tokenizers` library; `vocab_size=1000`, `min_frequency=5`
3. **Token selection** — diff BPE vocab vs mmBERT vocab; keep tokens with length >= 2
4. **Tokenizer extension** — `add_tokens()` adds new entries; vocab grows 256k → 256,637
5. **Embedding initialisation** — each new token's embedding is set to the mean of the original tokenizer's subword embeddings for that string (principled init; 0 random-only fallbacks)
6. **Verification** — reloads the saved model and prints before/after tokenisation for 10 Yoruba diacritic words

**Output:** `checkpoints/extended_model/` (extended tokenizer + base model weights)

In [ ]:
!python extend_vocab.py

EXTENSION B — Vocabulary Adaptation
  Base model:       jhu-clsp/mmBERT-small
  Target vocab add: 1000
  Min frequency:    5
  Output dir:       checkpoints/extended_model/

[Step 1] Collecting Yoruba corpus ...
  Sentences: 8,309  |  Words: 198,443

[Step 2] Training BPE tokeniser (vocab_size=1000, min_freq=5) ...
[00:00:00] Tokenize words                 ██████████████████ 19857    /    19857
[00:00:00] Count pairs                    ██████████████████ 19857    /    19857
[00:00:00] Compute merges                 ██████████████████ 855      /      855
  BPE vocab size after training: 1,000

[Step 3] Identifying new tokens ...
  BPE vocab size:      1,000
  mmBERT vocab size:   256,000
  New tokens (len>=2): 637
  First 20 examples:  ['19', '20', '201', '202', 'Adé', 'Alá', 'À', 'Ààrẹ', 'Àbújá', 'Àwọn', 'Àṣe', 'Àṣejèrè', 'Á', 'Bá', 'Bákan', 'Bí', 'Buha', 'Eag', 'È', 'Èkó']

[Step 4] Extending tokeniser with 637 new tokens ...
  Vocab before: 256,000
  Tokens added:

---
## Step 5: Tokenizer fertility analysis — after extension

**Script:** `analyse_tokenizer.py --tokenizer checkpoints/extended_model`

Runs the identical fertility analysis as Step 1 but on the extended tokenizer. Directly quantifies the improvement from vocabulary adaptation.

**What to look for:**
- Drop in `Mean toks/word` compared to Step 1 baseline
- Rise in `% single-token` words
- Diacritic probe: words like *ọmọ* and *ọjọ* should now appear as single tokens

**Output:** `results/tokenizer_analysis_extb.json`

In [ ]:
!python analyse_tokenizer.py --tokenizer checkpoints/extended_model


Loading tokenizer from: checkpoints/extended_model
Loading datasets...
Corpus: 8,309 sentences from news + NER train splits
Analysing 198,443 words ...

TOKENIZER FERTILITY — checkpoints/extended_model
Vocab size:          256,000
Words analysed:      198,443
Mean toks/word:      1.656
Median toks/word:    1.000
95th percentile:     4.000

Distribution:
  1_token     : 66.1%
  2_tokens    : 16.2%
  3_tokens    : 9.6%
  4+_tokens   : 8.1%

20 Most Fragmented Words:
  28 tokens  'ọ̀kẹ́méjìdínláàádọ́dalélẹ́gbààrúnlélárúndínláàádọ́ta'
  16 tokens  'òjìlélẹ́gbọ̀kànléláàádọ́ta'
  16 tokens  'òjìlélẹ́gbọ̀kànléláàádọ́ta'
  16 tokens  'o̩ló̩rè̩é̩jó̩rè̩é̩'
  16 tokens  'o̩ló̩rè̩é̩jó̩rè̩é̩'
  15 tokens  'Òjìlélẹ́ẹ̀ẹ́dẹ́gbèsándínméje'
  15 tokens  'Òjìlélẹ́ẹ̀ẹ́dẹ́gbèsándínmẹ́jọ'
  15 tokens  'o̩ló̩rè̩é̩jó̩rè̩é̩'
  15 tokens  'ọló̩rè̩é̩́jò̩ré̩'
  14 tokens  'ọlọ́rè̩é̩jò̩ré̩'
  13 tokens  'OJÚOLÚWAKÒ

---
## Step 6: Extension B training — MasakhaNews

**Script:** `train_news_extb.py`

Identical training procedure to Step 2 but uses the extended tokenizer and model from `checkpoints/extended_model/`. Same hyperparameters as the baseline to ensure a fair comparison — no re-tuning for the extended model.

The embedding matrix already has 256,637 rows with principled initialisations for the 637 new tokens. Only the classification head is freshly initialised.

**Output:** `checkpoints/news_extb/` and `results/news_extb.json`

In [ ]:
!python train_news_extb.py

Using device: cuda (Tesla T4)
TRAINING — MasakhaNews  [news_extb]
  model: checkpoints/extended_model/
  dataset: masakhane/masakhanews/yor
  max_seq_len: 128
  lr: 2e-05
  batch_size: 16
  epochs: 5
  warmup_steps: 100
  weight_decay: 0.01
  patience: 2
  seed: 42

Loading dataset ...
Labels (5): ['entertainment', 'health', 'politics', 'religion', 'sports']
Split sizes — train: 1433, val: 206, test: 411

Loading tokenizer + model from checkpoints/extended_model/ ...
Loading weights: 100% 134/134 [00:00<00:00, 1262.07it/s, Materializing param=model.layers.21.mlp_norm.weight]
ModernBertForSequenceClassification LOAD REPORT from: checkpoints/extended_model/
Key               | Status  | 
------------------+---------+-
head.dense.weight | MISSING | 
head.norm.weight  | MISSING | 
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Tokenising ...


---
## Step 7: Extension B training — MasakhaNER 2.0

**Script:** `train_ner_extb.py`

Identical training procedure to Step 3 but uses the extended vocabulary model. Lower tokenizer fertility means more complete sentences fit within the 128-token window, so the model may be evaluated on a slightly larger number of entities than the baseline (longer sentences are no longer truncated).

**Output:** `checkpoints/ner_extb/` and `results/ner_extb.json`

In [ ]:
!python train_ner_extb.py

Using device: cuda (Tesla T4)
TRAINING — MasakhaNER 2.0  [ner_extb]
  model: checkpoints/extended_model/
  dataset: masakhane/masakhaner2/yor
  max_seq_len: 128
  lr: 2e-05
  batch_size: 16
  epochs: 5
  warmup_steps: 100
  weight_decay: 0.01
  patience: 2
  seed: 42

Loading dataset ...
Labels (9): ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE']
Split sizes — train: 6876, val: 983, test: 1964

Loading tokenizer + model from checkpoints/extended_model/ ...
Loading weights: 100% 134/134 [00:00<00:00, 1265.43it/s, Materializing param=model.layers.21.mlp_norm.weight]
ModernBertForTokenClassification LOAD REPORT from: checkpoints/extended_model/
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
head.dense.weight | MISSING | 
classifier.bias   | MISSING | 
head.norm.weight  | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream tas

---
## Step 8: Test-set evaluation — initial run (LR = 2e-5)

**Script:** `evaluate.py`

Runs all four saved checkpoints on their respective held-out test sets. This is a read-only evaluation step — no model weights are updated.

> **Note:** These results are from the initial LR = 2e-5 run. Step 9 (LR sweep) subsequently identified 5e-5 as the optimal learning rate, and Step 10 retrains all models accordingly. **The Step 10 outputs are the final reported figures.**

In [ ]:
!python evaluate.py --checkpoint checkpoints/news_baseline --task news
!python evaluate.py --checkpoint checkpoints/ner_baseline  --task ner
!python evaluate.py --checkpoint checkpoints/news_extb     --task news
!python evaluate.py --checkpoint checkpoints/ner_extb      --task ner

Using device: cuda (Tesla T4)

Evaluating news checkpoint: /content/drive/MyDrive/csc5035z-a2/checkpoints/news_baseline
Loading weights: 100% 138/138 [00:00<00:00, 1546.56it/s, Materializing param=model.layers.21.mlp_norm.weight]
News eval: 100% 26/26 [00:07<00:00,  3.42it/s]
Test Macro-F1: 0.8424
               precision    recall  f1-score   support

entertainment       0.81      0.79      0.80       100
       health       0.86      0.81      0.83        80
     politics       0.84      0.86      0.85       100
     religion       0.74      0.80      0.77        64
       sports       0.97      0.96      0.96        67

     accuracy                           0.84       411
    macro avg       0.84      0.84      0.84       411
 weighted avg       0.84      0.84      0.84       411

Results saved -> results/eval_news_news_baseline.json
Using device: cuda (Tesla T4)

Evaluating NER checkpoint: /content/drive/MyDrive/csc5035z-a2/checkpoints/ner_baseline
Loading weights: 100% 138/138 [

In [ ]:
# Print results summary
import json, os, glob

print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)

for f in sorted(glob.glob('results/*.json')):
    name = os.path.basename(f).replace('.json', '')
    with open(f) as fp:
        data = json.load(fp)
    print(f'\n{name}:')
    for k, v in data.items():
        if isinstance(v, float):
            print(f'  {k}: {v:.4f}')
        elif k not in ('config', 'timestamp', 'test_predictions', 'test_true_labels',
                       'test_pred_tags', 'test_true_tags', 'per_class_f1', 'per_entity_f1'):
            print(f'  {k}: {v}')

RESULTS SUMMARY

eval_ner_ner_baseline:
  task: ner
  checkpoint: /content/drive/MyDrive/csc5035z-a2/checkpoints/ner_baseline
  test_span_f1: 0.8364
  label_list: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE']

eval_ner_ner_extb:
  task: ner
  checkpoint: /content/drive/MyDrive/csc5035z-a2/checkpoints/ner_extb
  test_span_f1: 0.8179
  label_list: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE']

eval_news_news_baseline:
  task: news
  checkpoint: /content/drive/MyDrive/csc5035z-a2/checkpoints/news_baseline
  test_macro_f1: 0.8424
  label_names: ['entertainment', 'health', 'politics', 'religion', 'sports']

eval_news_news_extb:
  task: news
  checkpoint: /content/drive/MyDrive/csc5035z-a2/checkpoints/news_extb
  test_macro_f1: 0.8469
  label_names: ['entertainment', 'health', 'politics', 'religion', 'sports']

ner_baseline:
  best_val_span_f1: 0.7887
  test_span_f1: 0.8364
  label_list: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-O

---
## Step 9: Learning rate sweep

**Script:** `sweep_lr.py`

Sweeps four LR candidates — {1e-5, 2e-5, 3e-5, 5e-5} — on both tasks using **train and validation splits only**. The test set is never seen during the sweep.

Each trial trains for up to 5 epochs with early stopping (patience = 2) and reports the best validation F1. No checkpoints are saved — this is a hyperparameter selection step only.

**Result:** LR = 5e-5 achieved the best validation F1 on both tasks, outperforming the 2e-5 used in Steps 1–8. All models are retrained in Step 10.

**Output:** `results/lr_sweep.json`

In [ ]:
!python sweep_lr.py

Using device: cuda (Tesla T4)
LR SWEEP
  LR values: [1e-05, 2e-05, 3e-05, 5e-05]
  Tasks: news, ner
  Max epochs: 5, patience: 2

--- News  LR=1e-05 ---
README.md: 10.7kB [00:00, 22.7MB/s]
train.tsv: 3.78MB [00:00, 46.5MB/s]
dev.tsv: 517kB [00:00, 103MB/s]
test.tsv: 1.07MB [00:00, 20.0MB/s]
Generating train split: 100% 1433/1433 [00:00<00:00, 11140.20 examples/s]
Generating validation split: 100% 206/206 [00:00<00:00, 10829.71 examples/s]
Generating test split: 100% 411/411 [00:00<00:00, 11351.48 examples/s]
config.json: 1.19kB [00:00, 3.51MB/s]
tokenizer_config.json: 46.4kB [00:00, 88.7MB/s]
tokenizer.json: 100% 17.5M/17.5M [00:00<00:00, 25.2MB/s]
special_tokens_map.json: 100% 636/636 [00:00<00:00, 3.54MB/s]
pytorch_model.bin: 100% 564M/564M [00:04<00:00, 140MB/s]
Loading weights: 100% 136/136 [00:00<00:00, 1615.08it/s, Materializing param=model.layers.21.mlp_norm.weight]
ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-small
Key               | Status     | 
----

---
## Step 10: Retrain all models with optimal LR (5e-5)

The sweep identified **LR = 5e-5** as best for both tasks:

| LR | News val-F1 | NER val-F1 |
|---|---|---|
| 1e-5 | 0.7983 | 0.7446 |
| 2e-5 | 0.8414 | 0.7791 |
| 3e-5 | 0.8293 | 0.8198 |
| **5e-5** | **0.8533** | **0.8198** |

`config.py` was updated to `LEARNING_RATE = 5e-5`. All four models are retrained from scratch and re-evaluated on the test set. These are the final reported results.

In [ ]:
!python train_news.py

Using device: cuda (Tesla T4)
TRAINING — MasakhaNews  [news_baseline]
  model: jhu-clsp/mmBERT-small
  dataset: masakhane/masakhanews/yor
  max_seq_len: 128
  lr: 5e-05
  batch_size: 16
  epochs: 5
  warmup_steps: 100
  weight_decay: 0.01
  patience: 2
  seed: 42

Loading dataset ...
Labels (5): ['entertainment', 'health', 'politics', 'religion', 'sports']
Split sizes — train: 1433, val: 206, test: 411

Loading tokenizer + model from jhu-clsp/mmBERT-small ...
Loading weights: 100% 136/136 [00:00<00:00, 1274.85it/s, Materializing param=model.layers.21.mlp_norm.weight]
ModernBertForSequenceClassification LOAD REPORT from: jhu-clsp/mmBERT-small
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params w

In [ ]:
!python train_ner.py

Using device: cuda (Tesla T4)
TRAINING — MasakhaNER 2.0  [ner_baseline]
  model: jhu-clsp/mmBERT-small
  dataset: masakhane/masakhaner2/yor
  max_seq_len: 128
  lr: 5e-05
  batch_size: 16
  epochs: 5
  warmup_steps: 100
  weight_decay: 0.01
  patience: 2
  seed: 42

Loading dataset ...
Labels (9): ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE']
Split sizes — train: 6876, val: 983, test: 1964

Loading tokenizer + model from jhu-clsp/mmBERT-small ...
Loading weights: 100% 136/136 [00:00<00:00, 1742.33it/s, Materializing param=model.layers.21.mlp_norm.weight]
ModernBertForTokenClassification LOAD REPORT from: jhu-clsp/mmBERT-small
Key               | Status     | 
------------------+------------+-
decoder.weight    | UNEXPECTED | 
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSI

In [ ]:
!python train_news_extb.py

Using device: cuda (Tesla T4)
TRAINING — MasakhaNews  [news_extb]
  model: checkpoints/extended_model/
  dataset: masakhane/masakhanews/yor
  max_seq_len: 128
  lr: 5e-05
  batch_size: 16
  epochs: 5
  warmup_steps: 100
  weight_decay: 0.01
  patience: 2
  seed: 42

Loading dataset ...
Labels (5): ['entertainment', 'health', 'politics', 'religion', 'sports']
Split sizes — train: 1433, val: 206, test: 411

Loading tokenizer + model from checkpoints/extended_model/ ...
Loading weights: 100% 134/134 [00:06<00:00, 20.01it/s, Materializing param=model.layers.21.mlp_norm.weight]
ModernBertForSequenceClassification LOAD REPORT from: checkpoints/extended_model/
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
head.norm.weight  | MISSING | 
head.dense.weight | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Tokenising ...

S

In [ ]:
!python train_ner_extb.py

Using device: cuda (Tesla T4)
TRAINING — MasakhaNER 2.0  [ner_extb]
  model: checkpoints/extended_model/
  dataset: masakhane/masakhaner2/yor
  max_seq_len: 128
  lr: 5e-05
  batch_size: 16
  epochs: 5
  warmup_steps: 100
  weight_decay: 0.01
  patience: 2
  seed: 42

Loading dataset ...
Labels (9): ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-DATE', 'I-DATE']
Split sizes — train: 6876, val: 983, test: 1964

Loading tokenizer + model from checkpoints/extended_model/ ...
Loading weights: 100% 134/134 [00:00<00:00, 258.07it/s, Materializing param=model.layers.21.mlp_norm.weight]
ModernBertForTokenClassification LOAD REPORT from: checkpoints/extended_model/
Key               | Status  | 
------------------+---------+-
head.dense.weight | MISSING | 
classifier.weight | MISSING | 
classifier.bias   | MISSING | 
head.norm.weight  | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task

In [7]:
!python evaluate.py --checkpoint checkpoints/news_baseline --task news
!python evaluate.py --checkpoint checkpoints/ner_baseline  --task ner
!python evaluate.py --checkpoint checkpoints/news_extb     --task news
!python evaluate.py --checkpoint checkpoints/ner_extb      --task ner

Using device: cuda (Tesla T4)

Evaluating news checkpoint: /content/drive/MyDrive/csc5035z-a2/checkpoints/news_baseline
Loading weights: 100% 138/138 [00:00<00:00, 570.90it/s, Materializing param=model.layers.21.mlp_norm.weight]
README.md: 10.7kB [00:00, 22.3MB/s]
train.tsv: 3.78MB [00:00, 84.2MB/s]
dev.tsv: 517kB [00:00, 140MB/s]
test.tsv: 1.07MB [00:00, 182MB/s]
Generating train split: 100% 1433/1433 [00:00<00:00, 15302.81 examples/s]
Generating validation split: 100% 206/206 [00:00<00:00, 14374.81 examples/s]
Generating test split: 100% 411/411 [00:00<00:00, 15841.23 examples/s]
News eval:   0% 0/26 [00:00<?, ?it/s]W0520 18:31:18.202000 8212 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode
News eval: 100% 26/26 [00:13<00:00,  1.89it/s]
Test Macro-F1: 0.8641
               precision    recall  f1-score   support

entertainment       0.88      0.78      0.83       100
       health       0.83      0.88      0.85        80
     politics       0.82    

---
## Experiment complete

All results saved to `results/`. Open `notebooks/results_analysis.ipynb` to generate summary tables, figures, and error analysis.

### Final results (LR = 5e-5)
| Model | MasakhaNews Macro-F1 | MasakhaNER Span-F1 |
|---|---|---|
| mmBERT-small Baseline | **0.8641** | **0.8407** |
| mmBERT-small + Vocab Ext (Ext B) | 0.8605 | 0.8393 |
| Delta | −0.0036 | −0.0014 |

**Key finding:** Vocabulary adaptation reduced tokenizer fertility by 47% (3.12 → 1.66 mean tokens/word) but had negligible downstream effect at the optimal LR. The NER regression seen at 2e-5 was a learning rate artefact.